# Lab05 — Bài 8. Connect 4 (bản hoàn thiện và kiểm thử)

**Môn:** Trí tuệ nhân tạo nâng cao — Tìm kiếm đối kháng và trò chơi.

Notebook này sử dụng `connect4_lab05.py` trong cùng thư mục, gồm mô hình trò chơi, Random Agent, Minimax, Alpha-Beta, Move Ordering, heuristic cutoff và PMCS. **Số liệu bên dưới được thực thi từ mã mới**, không sao chép số liệu trước đó của Google Docs. Test thực hiện trên môi trường Python hiện tại; tốc độ chạy thay đổi theo phần cứng.


## Task 1. Mô hình và độ phức tạp

- **Trạng thái**: ma trận `R×C`, 0 là trống; 1 và -1 là quân của hai người chơi. Quân rơi xuống ô trống thấp nhất trong cột.
- **Actions(s)**: các cột chưa đầy; **Result(s,p,a)**: tạo bản sao bảng, thả quân `p` vào cột `a`.
- **Terminal(s)**: xuất hiện 4 quân liên tục theo ngang/dọc/chéo hoặc hết chỗ; **Utility(s)** theo người chơi Max là +1 thắng, -1 thua, 0 hòa, None chưa kết thúc.
- **Không gian trạng thái**: cận trên đơn giản `3^(R*C)` (không phải mọi ma trận đều hợp lệ vì luật rơi quân/đổi lượt). **Cây trò chơi** có hệ số phân nhánh ≤ C và độ sâu ≤ R*C, nên cận trên rất lỏng là `O(C^(R*C))`. Việc ghi nhớ trạng thái và pruning thay đổi số nút thực tế.
- **Bàn chuẩn 6×7**: không áp dụng Minimax tìm tới lá trong thời gian phản hồi thông thường; dùng cutoff/heuristic hoặc simulation. Các kết luận về thời gian phải được đo với code và phần cứng cụ thể.


In [1]:
import numpy as np, pandas as pd
from connect4_lab05 import (empty_board, actions, result, utility, terminal, winners,
  random_player, minimax_search, alpha_beta_search, ordered_actions,
  eval_fun, pmcs, play, Connect4Agent)
np.set_printoptions(linewidth=90)
print('Đã nạp module Connect 4, NumPy', np.__version__)

Đã nạp module Connect 4, NumPy 2.3.5


## Task 2. Môi trường, hành động hợp lệ, Random Agent

In [2]:
b=empty_board((4,4))
print('Bàn rỗng:', '\n', b)
print('Nước hợp lệ:', actions(b))
for p,c in ((1,1),(-1,1),(1,0)):
    b=result(b,p,c)
print('Bàn sau 3 bước:\n',b)
print('Bàn gốc không bị thay đổi khi result:', np.count_nonzero(empty_board((4,4)))==0)
print('Kết thúc?',terminal(b),'Utility(X):',utility(b,1))

Bàn rỗng: 
 [[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
Nước hợp lệ: [0, 1, 2, 3]
Bàn sau 3 bước:
 [[ 0  0  0  0]
 [ 0  0  0  0]
 [ 0 -1  0  0]
 [ 1  1  0  0]]
Bàn gốc không bị thay đổi khi result: True
Kết thúc? False Utility(X): None


In [3]:
r_rand, per_move_random=play(Connect4Agent('random',seed=42),Connect4Agent('random',seed=43),games=1000,seed=42,shape=(4,4))
print('Random X vs Random O: 1000 ván trên 4x4:',r_rand)
print('Thời gian gọi agent X trung bình (ms):',round(per_move_random,4))
assert sum(r_rand.values()) == 1000

Random X vs Random O: 1000 ván trên 4x4: {'x': 267, 'o': 245, 'd': 488}
Thời gian gọi agent X trung bình (ms): 0.0136


## Task 3. Minimax / Alpha-Beta và ít nhất 5 trạng thái kiểm thử

Các trạng thái được dựng tuân theo trọng lực. Dùng vị trí có khả năng thắng/chặn ngay để kiểm tra nước đi, rồi dùng trạng thái nhỏ để kiểm tra giá trị Minimax/Alpha-Beta. Có nhiều nước đi tối ưu đồng hạng, nên so sánh giá trị tối ưu và tính hợp lệ thay vì mặc định mọi thuật toán phải trả cùng chỉ số cột.

In [4]:
def from_moves(seq, shape=(4,4)):
    b=empty_board(shape)
    for p,c in seq:
        b=result(b,p,c)
    return b
boards={
 'X dọc thắng ngay':from_moves([(1,0),(-1,1),(1,0),(-1,1),(1,0),(-1,2)]),
 'X ngang thắng ngay':from_moves([(1,0),(-1,0),(1,1),(-1,1),(1,2),(-1,2)]),
 'Chặn O cột 3':from_moves([(-1,3),(1,0),(-1,3),(1,2),(-1,3),(1,0)]),
 'Giữa ván A':from_moves([(1,0),(-1,1),(1,0),(-1,1),(1,2),(-1,2)]),
 'Giữa ván B':from_moves([(1,2),(-1,1),(1,3),(-1,1),(1,2),(-1,0)])}
rows=[]
for name,state in boards.items():
    ans=alpha_beta_search(state,player=1,cutoff=3,move_ordering=True)
    assert ans.move in actions(state)
    rows.append({'Trạng thái':name,'Nước đi AB':ans.move,'Giá trị':ans.value,'Nút':ans.nodes,'Thời gian (ms)':round(ans.elapsed_ms,2)})
display(pd.DataFrame(rows))
assert rows[0]['Nước đi AB']==0
assert rows[1]['Nước đi AB']==3
assert rows[2]['Nước đi AB']==3

,Trạng thái,Nước đi AB,Giá trị,Nút,Thời gian (ms)
0,X dọc thắng ngay,0,1000000.0,17,2.64
1,X ngang thắng ngay,3,1000000.0,20,2.55
2,Chặn O cột 3,3,25.5,20,2.49
3,Giữa ván A,0,23.5,48,5.98
4,Giữa ván B,2,85.5,38,4.45


In [5]:
same=boards['Giữa ván A']
mi=minimax_search(same,player=1,cutoff=None)
ab=alpha_beta_search(same,player=1,cutoff=None,move_ordering=False)
print('Minimax không cắt tỉa:',mi)
print('Alpha-Beta, cùng trạng thái:',ab)
assert mi.value == ab.value
assert ab.nodes <= mi.nodes
assert mi.move in actions(same) and ab.move in actions(same)
print('Kiểm tra giá trị tối ưu: PASS')

Minimax không cắt tỉa: SearchResult(move=0, value=0.0, nodes=40609, elapsed_ms=1378.8174779999736)
Alpha-Beta, cùng trạng thái: SearchResult(move=0, value=0.0, nodes=442, elapsed_ms=13.232390999974086)
Kiểm tra giá trị tối ưu: PASS


### Move Ordering

Thứ tự ưu tiên: nước thắng ngay → nước chặn đối phương thắng ngay → cột giữa. Kết quả pruning có thể tăng/giảm tùy trạng thái; không suy diễn mức tiết kiệm >80% khi chưa đo.

In [6]:
test_state=boards['Giữa ván B']
no_order=alpha_beta_search(test_state,1,cutoff=5,move_ordering=False)
with_order=alpha_beta_search(test_state,1,cutoff=5,move_ordering=True)
print('Không ordering:', no_order)
print('Có ordering:', with_order)
print('Sai khác nút:',with_order.nodes-no_order.nodes)
assert no_order.value == with_order.value

Không ordering: SearchResult(move=2, value=88.0, nodes=413, elapsed_ms=32.40827599984186)
Có ordering: SearchResult(move=2, value=88.0, nodes=163, elapsed_ms=20.217172999991817)
Sai khác nút: -250


## Task 4. Heuristic Alpha-Beta với cutoff

Ở leaf chưa terminal, chấm điểm cửa sổ 4 ô, phạt những mối đe dọa của đối thủ và thưởng kiểm soát trung tâm. Trạng thái terminal dùng giá trị lớn ±1.000.000, ưu tiên tuyệt đối so với heuristic thông thường. So sánh cutoff trên **cùng trạng thái**.

In [7]:
b=empty_board((6,7))
for p,c in [(1,3),(-1,2),(1,3),(-1,4)]:b=result(b,p,c)
comparison=[]
for cutoff in (2,4):
    res=alpha_beta_search(b,1,cutoff=cutoff,move_ordering=True)
    comparison.append({'cutoff':cutoff,'move':res.move,'value':res.value,'nodes':res.nodes,'time_ms':round(res.elapsed_ms,2)})
display(pd.DataFrame(comparison))
assert all(row['move'] in actions(b) for row in comparison)

,cutoff,move,value,nodes,time_ms
0,2,2,43.0,28,16.08
1,4,3,55.0,202,96.72


## Task 5. Pure Monte Carlo Search

`N` được định nghĩa là **tổng ngân sách mô phỏng**, chia gần đều giữa các cột hợp lệ; khác với cách hiểu N mô phỏng *cho mỗi cột*. Mỗi lựa chọn sử dụng một generator với seed xác định. PMCS có biến thiên ngẫu nhiên, cần lặp nhiều seeds để ước lượng độ ổn định.

In [8]:
mc_results=[]
for N in (10,100,1000):
    from time import perf_counter
    t=perf_counter()
    move=pmcs(b,1,N=N,rng=np.random.default_rng(42))
    mc_results.append({'N':N,'move':move,'time_ms':round((perf_counter()-t)*1000,2)})
display(pd.DataFrame(mc_results))
assert all(x['move'] in actions(b) for x in mc_results)

,N,move,time_ms
0,10,3,12.71
1,100,2,109.98
2,1000,3,1119.99


## Đấu thử và kết luận có giới hạn

Mỗi agent dùng seed riêng; seed của environment là cố định, nhưng **các phương pháp có đường gọi random khác nhau không được xem như cùng chuỗi tình huống**. Benchmark đầy đủ cần nhiều seed, cùng tiêu chí chọn nước, thống kê độ lệch và phần cứng ghi rõ.

In [9]:
runs=[]
for name,ax in [('Heuristic-AB c2',Connect4Agent('alpha_beta',cutoff=2)),('PMCS N=20',Connect4Agent('pmcs',N=20,seed=42))]:
    result_counts,mean_time=play(ax,Connect4Agent('random',seed=7),games=12,shape=(4,4),seed=42)
    runs.append({'agent':name,**result_counts,'N ván':12,'ms/nước X':round(mean_time,2)})
display(pd.DataFrame(runs))
assert all(r['x']+r['o']+r['d']==12 for r in runs)

,agent,x,o,d,N ván,ms/nước X
0,Heuristic-AB c2,5,0,7,12,1.40
1,PMCS N=20,11,0,1,12,6.98


### Nhận xét dựa trên code và phép đo ở trên

1. Minimax đầy đủ có thể giải các thế cờ nhỏ nhưng rất tốn số nút; so với Minimax, Alpha-Beta giữ nguyên giá trị tối ưu khi cùng thứ tự/leaf evaluation.
2. Move Ordering không thay đổi giá trị tối ưu nếu search đầy đủ và cùng hàm đánh giá; hiệu quả cắt tỉa phụ thuộc thế cờ, không bảo đảm tốt hơn ở tất cả trạng thái.
3. Cutoff nông giảm số tầng tìm kiếm, nhưng heuristic có thể đánh giá sai các mối đe dọa nằm ngoài horizon.
4. Monte Carlo chỉ cần hàm luật chơi và mô phỏng, nhưng kết quả có nhiễu lấy mẫu, và tăng N làm tăng thời gian tính toán.
5. Kết quả thực nghiệm trên 4×4 không chuyển nguyên tỷ lệ sang bàn 6×7; cần benchmark riêng trên bàn chuẩn và giới hạn tài nguyên.

**Phạm vi:** Notebook hoàn thiện phần Connect 4 theo bài mở rộng; **không khẳng định đã hoàn thành Mean Connect 4 hoặc Dots and Boxes**. Để lập báo cáo tổng hợp các thuật toán Tic-Tac-Toe, phải dùng bộ thí nghiệm riêng sau khi sửa AND-OR.
